In [ ]:
from pathlib import Path
import subprocess, sys
Path('/kaggle/working/t092_worker.py').write_text('"""T092: Super-enriched row-only XGBoost (d5/d6/d7) + CatBoost GPU (d6/d8 with low-card cats)."""\nimport gc, json, sys, time\nfrom pathlib import Path\nimport numpy as np\nimport polars as pl\nimport pyarrow.parquet as pq\nfrom scipy.special import logit\nfrom sklearn.metrics import roc_auc_score\n\nOUT = Path(\'/kaggle/working/t092\')\nCACHE = Path(\'/kaggle/temp/t092\')\nPRIMARY_CUT = np.datetime64(\'2026-10-28T08:06:52.397\')\nEARLIER_CUT = np.datetime64(\'2026-10-27T08:06:52.397\')\nEARLIER_END = np.datetime64(\'2026-10-27T18:00:00\')\nLOW_CATS = [f\'feature{i}\' for i in [6,7,8,9,10,11,12,13,14,16,17,19,21,22,25]]\nHIST_DUR = [476,477,479,480,483,484,486,487,492,493,500,501,502,503,512,513,514,515]\nVEC_COLS = [f\'feature{i}\' for i in [*range(26,33), *range(40,47)]]\nUSER_RATES = [\'feature352\', \'feature346\', \'feature358\', \'feature334\']\nITEM_RATES = [\'feature418\', \'feature419\', \'feature430\', \'feature411\']\n\n\ndef numeric_cols(schema):\n    return [\'feature3\'] + [\n        c for c, t in schema.items()\n        if c.startswith(\'feature\') and c not in (\'feature1\', \'feature3\')\n        and str(t) in (\'Float32\', \'Int64\', \'Int8\')\n    ]\n\n\ndef safe_logit(a):\n    return logit(np.clip(np.where(np.isfinite(a), a, np.nan), 1e-5, 1.0 - 1e-5))\n\n\ndef super_features(f, num_cols):\n    n = len(f)\n    raw = np.column_stack([f[c].to_numpy().astype(\'f4\') for c in num_cols])\n    vecs = np.stack([f[c].fill_null([0.0]*32).list.to_array(32).to_numpy().astype(\'f4\') for c in VEC_COLS], axis=1)\n    norms = np.linalg.norm(vecs, axis=2)\n    units = vecs / np.maximum(norms[:, :, None], 1e-6)\n    bias = np.column_stack([f[f\'feature{i}\'].to_numpy().astype(\'f4\') for i in [*range(33,40), *range(47,54)]])\n    dots = np.einsum(\'bik,bjk->bij\', vecs[:, :7], vecs[:, 7:]) + bias[:, :7, None] + bias[:, None, 7:]\n    cos49 = np.einsum(\'bik,bjk->bij\', units[:, :7], units[:, 7:])\n    # 36 duration interactions\n    d = f[\'feature54\'].to_numpy().astype(\'d\')\n    d = np.where(d > 0, d, np.nan)\n    s = np.column_stack([f[f\'feature{i}\'].to_numpy().astype(\'d\') for i in HIST_DUR])\n    s = np.where(s >= 0, s, np.nan)\n    dur = np.column_stack([np.log1p(s) - np.log1p(d[:, None]), s - d[:, None]]).astype(\'f4\')\n    # Stationary ages + time-of-day\n    now = f[\'feature1\'].dt.epoch(\'ms\').to_numpy().astype(\'d\') / 1000.0\n    st57 = f[\'feature57\'].to_numpy().astype(\'d\')\n    st57 = np.where(st57 > 1e11, st57 / 1000.0, st57)\n    st58 = f[\'feature58\'].to_numpy().astype(\'d\')\n    st59 = f[\'feature59\'].to_numpy().astype(\'d\') / 1000.0\n    ages = [np.log1p(np.where((st > 0) & (now >= st), (now - st) / 86400.0, np.nan)) for st in (st57, st58, st59)]\n    hr = (now % 86400.0) / 86400.0\n    # User x Item watch-rate interactions (32 features: product + logit sum)\n    ur = [f[c].to_numpy().astype(\'d\') for c in USER_RATES]\n    ir = [f[c].to_numpy().astype(\'d\') for c in ITEM_RATES]\n    rate_cross = []\n    for u in ur:\n        lu = safe_logit(u)\n        for v in ir:\n            rate_cross.append(u * v)\n            rate_cross.append(lu + safe_logit(v))\n    # Top cosine x rate/duration interactions (11 features)\n    c32_43 = cos49[:, 6, 3].astype(\'d\')\n    c30_46 = cos49[:, 4, 6].astype(\'d\')\n    l418 = safe_logit(ir[0])\n    l352 = safe_logit(ur[0])\n    ld = np.log1p(d)\n    l493 = np.log1p(np.maximum(f[\'feature493\'].to_numpy().astype(\'d\'), 0))\n    cos_cross = [\n        c32_43 * l418, c32_43 * l352, c32_43 * ld, c32_43 * l493,\n        c30_46 * l418, c30_46 * l352, c30_46 * ld, c30_46 * l493,\n        c32_43 + c30_46, c32_43 - c30_46, c32_43 * c30_46,\n    ]\n    # User watch-time vs item watch-time log ratios (9 features)\n    uwt = [np.log1p(np.maximum(f[f\'feature{k}\'].to_numpy().astype(\'d\'), 0)) for k in (493, 480, 484)]\n    iwt = [np.log1p(np.maximum(f[f\'feature{k}\'].to_numpy().astype(\'d\'), 0)) for k in (515, 511, 502)]\n    wt_cross = [u - v for u in uwt for v in iwt]\n    # Watch-to-skip ratios (3 features)\n    ratios = [\n        ur[0] / (np.maximum(ur[2], 0) + 1e-4),\n        ur[1] / (np.maximum(ur[3], 0) + 1e-4),\n        ir[0] / (np.maximum(ir[2], 0) + 1e-4),\n    ]\n    extra = np.column_stack([*ages, np.sin(2*np.pi*hr), np.cos(2*np.pi*hr), *rate_cross, *cos_cross, *wt_cross, *ratios]).astype(\'f4\')\n    return np.column_stack([raw, vecs.reshape(n, 448), norms, dots.reshape(n, 49), cos49.reshape(n, 49), dur, extra]).astype(\'f4\')\n\n\ndef prepare(raw_path):\n    schema = pl.scan_parquet(raw_path).collect_schema()\n    num_cols = numeric_cols(schema)\n    assert \'request_id\' not in num_cols and len(num_cols) == 584\n    fields = [\'feature1\'] + num_cols + VEC_COLS\n    pf = pq.ParquetFile(raw_path)\n    n = pf.metadata.num_rows\n    assert n == 1537638\n    width = 584 + 448 + 14 + 49 + 49 + 36 + 60\n    x = np.lib.format.open_memmap(CACHE/\'x.npy\', mode=\'w+\', dtype=\'f4\', shape=(n, width))\n    off = 0\n    for batch in pf.iter_batches(columns=fields, batch_size=8192):\n        f = pl.from_arrow(batch)\n        z = super_features(f, num_cols)\n        if off == 0:\n            z1 = super_features(f.slice(0, 1), num_cols)\n            np.testing.assert_allclose(z[:1], z1, atol=1e-5, equal_nan=True)\n        x[off:off+len(f)] = z\n        off += len(f)\n    assert off == n\n    x.flush()\n    meta = pl.read_parquet(raw_path, columns=[\'feature1\', \'watched\'])\n    np.save(CACHE/\'t.npy\', meta[\'feature1\'].to_numpy())\n    np.save(CACHE/\'y.npy\', meta[\'watched\'].to_numpy())\n    print(f\'Prepared super matrix {x.shape}\', flush=True)\n\n\ndef get_split(fold):\n    t = np.load(CACHE/\'t.npy\'); y = np.load(CACHE/\'y.npy\')\n    cut = PRIMARY_CUT if fold == \'primary\' else EARLIER_CUT\n    tr = np.flatnonzero(t < cut)\n    va = np.flatnonzero((t >= cut) & ((fold == \'primary\') | (t < EARLIER_END)))\n    assert (len(tr), len(va)) == ((1230110, 307528) if fold == \'primary\' else (710330, 299969))\n    return tr, va, y\n\n\ndef fit_xgb(fold, cfg):\n    import cupy as cp\n    import xgboost as xgb\n    tr, va, y = get_split(fold)\n    x = np.load(CACHE/\'x.npy\', mmap_mode=\'r\')\n    specs = {\n        \'xgb_d6\': (dict(max_depth=6, eta=0.02, colsample_bytree=0.4, colsample_bynode=0.5, min_child_weight=150, reg_lambda=20, reg_alpha=2, gamma=1, seed=42), (900, 1200, 1500)),\n        \'xgb_d5\': (dict(max_depth=5, eta=0.02, colsample_bytree=0.35, colsample_bynode=0.5, min_child_weight=200, reg_lambda=30, reg_alpha=5, gamma=1, seed=43), (1200, 1600, 2000)),\n        \'xgb_d7\': (dict(max_depth=7, eta=0.02, colsample_bytree=0.4, colsample_bynode=0.4, min_child_weight=150, reg_lambda=25, reg_alpha=3, gamma=1, seed=44), (900, 1200)),\n    }\n    p_cfg, rounds_list = specs[cfg]\n    params = dict(objective=\'binary:logistic\', tree_method=\'hist\', device=\'cuda\', subsample=0.8, max_bin=127, nthread=4, **p_cfg)\n\n    class Iter(xgb.DataIter):\n        def __init__(self): self.reset(); super().__init__()\n        def reset(self): self.i = 0\n        def next(self, input_data):\n            if self.i >= len(tr): return False\n            j = tr[self.i:self.i+8192]; self.i += 8192\n            input_data(data=cp.asarray(x[j]), label=cp.asarray(y[j]))\n            return True\n\n    dm = xgb.QuantileDMatrix(Iter(), max_bin=127, nthread=4)\n    model = xgb.train(params, dm, max(rounds_list))\n    del dm; gc.collect(); cp.get_default_memory_pool().free_all_blocks()\n    res = {}\n    for r in rounds_list:\n        def pred(j): return cp.asnumpy(model.inplace_predict(cp.asarray(x[j]), iteration_range=(0, r)))\n        p = np.concatenate([pred(va[i:i+8192]) for i in range(0, len(va), 8192)]).astype(\'f4\')\n        np.testing.assert_allclose(p[:8], np.concatenate([pred(j[None]) for j in va[:8]]), atol=1e-6)\n        auc = float(roc_auc_score(y[va], p))\n        res[str(r)] = auc\n        print(f\'{fold} {cfg} r={r} auc={auc:.6f}\', flush=True)\n        np.savez_compressed(OUT/f\'{fold}-{cfg}-r{r}.npz\', row_ids=va, y_true=y[va], probabilities=p)\n    return res\n\n\ndef fit_cat(raw_path, fold, cfg):\n    from catboost import CatBoostClassifier, FeaturesData, Pool\n    tr, va, y = get_split(fold)\n    x = np.load(CACHE/\'x.npy\', mmap_mode=\'r\')\n    # Only low-cardinality categoricals (<= 1000 unique in train, strictly excluding user/item/creator IDs)\n    df_cat = pl.read_parquet(raw_path, columns=LOW_CATS)\n    keep_cats = []\n    cat_cols_arr = []\n    for col in LOW_CATS:\n        s = df_cat[col]\n        if s.gather(tr).drop_nulls().n_unique() <= 1000:\n            keep_cats.append(col)\n            cat_cols_arr.append(s.fill_null(\'__NA__\').to_numpy().astype(str))\n    cat_mat = np.column_stack(cat_cols_arr).astype(object)\n    del df_cat, cat_cols_arr; gc.collect()\n\n    depth, iters, l2, seed = (6, 3000, 15.0, 42) if cfg == \'cat_d6\' else (8, 2200, 25.0, 43)\n    tr_pool = Pool(FeaturesData(num_feature_data=np.ascontiguousarray(x[tr], dtype=np.float32), cat_feature_data=cat_mat[tr]), label=y[tr], thread_count=4)\n    model = CatBoostClassifier(\n        iterations=iters, learning_rate=0.03, depth=depth, l2_leaf_reg=l2,\n        border_count=128, random_seed=seed, task_type=\'GPU\', devices=\'0\',\n        gpu_ram_part=0.85, max_ctr_complexity=2, verbose=500, allow_writing_files=False,\n    )\n    model.fit(tr_pool)\n    del tr_pool; gc.collect()\n\n    va_pool = Pool(FeaturesData(num_feature_data=np.ascontiguousarray(x[va], dtype=np.float32), cat_feature_data=cat_mat[va]), thread_count=4)\n    p = model.predict_proba(va_pool)[:, 1].astype(\'f4\')\n    p1 = model.predict_proba(Pool(FeaturesData(num_feature_data=np.ascontiguousarray(x[va[:1]], dtype=np.float32), cat_feature_data=cat_mat[va[:1]]), thread_count=4))[:, 1].astype(\'f4\')\n    np.testing.assert_allclose(p[:1], p1, atol=1e-6)\n    auc = float(roc_auc_score(y[va], p))\n    print(f\'{fold} {cfg} iters={iters} cats={len(keep_cats)} auc={auc:.6f}\', flush=True)\n    np.savez_compressed(OUT/f\'{fold}-{cfg}.npz\', row_ids=va, y_true=y[va], probabilities=p)\n    del va_pool, model, cat_mat; gc.collect()\n    return {\'auc\': auc, \'cats\': keep_cats}\n\n\nif __name__ == \'__main__\':\n    OUT.mkdir(exist_ok=True, parents=True); CACHE.mkdir(exist_ok=True, parents=True)\n    raw = next(p for p in Path(\'/kaggle/input\').rglob(\'train.parquet\') if (p.parent/\'sample_submission.csv\').is_file())\n    args = sys.argv[1:]; t0 = time.monotonic()\n    if args == [\'prepare\']:\n        prepare(raw); res = {}\n    elif args[1].startswith(\'xgb\'):\n        res = fit_xgb(args[0], args[1])\n    else:\n        res = fit_cat(raw, args[0], args[1])\n    res[\'seconds\'] = time.monotonic() - t0\n    (OUT/(\'-\'.join(args)+\'.json\')).write_text(json.dumps(res) + \'\\n\')\n')
stages = [['prepare']] + [
    [fold, cfg]
    for fold in ['earlier', 'primary']
    for cfg in ['xgb_d6', 'xgb_d5', 'xgb_d7', 'cat_d6', 'cat_d8']
]
for args in stages:
    print('=== STAGE', args, flush=True)
    subprocess.run([sys.executable, '-u', '/kaggle/working/t092_worker.py', *args], check=True)
